# 02 - Evaluation Pipeline
## AI Chatbot Evaluation Framework
**CSE 533 - Machine Learning** | MD Raihan Khan & Zannatul Islam Proma

This notebook runs the NLP evaluation pipeline:
1. Load responses and ground truth
2. Run NLP evaluation (keyword matching, NER, semantic similarity)
3. Compute accuracy, authenticity, and up-to-dateness scores

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import json
import pandas as pd

from src.query_manager import QueryManager
from src.chatbot_interface import ChatbotInterface
from src.ground_truth_collector import GroundTruthCollector
from src.nlp_evaluator import NLPEvaluator
from src.scorer import Scorer

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

qm = QueryManager('../data/queries/evaluation_queries.json')
queries = qm.get_all_queries()
print(f'Loaded {len(queries)} queries')

## 1. Load Responses and Ground Truth

In [ ]:
import os

ci = ChatbotInterface(config)

# Try loading API responses first, then manual
api_path = '../data/responses/chatbot_responses.json'
manual_path = '../data/responses/manual_collection_template.json'

if os.path.exists(api_path):
    responses = ci.load_responses(api_path)
    print(f'Loaded API responses: {len(responses)} queries')
elif os.path.exists(manual_path):
    responses = ci.load_manual_responses(manual_path)
    print(f'Loaded manual responses: {len(responses)} queries')
else:
    print('No responses found! Run notebook 01 first.')

In [ ]:
# Load ground truth
gtc = GroundTruthCollector(config)
gt_path = '../data/ground_truth/ground_truth_data.json'

if os.path.exists(gt_path):
    gt_data = gtc.load_ground_truth(gt_path)
else:
    gt_data = gtc.extract_from_queries(queries)

print(f'Ground truth entries: {len(gt_data)}')

## 2. Run NLP Evaluation

In [ ]:
evaluator = NLPEvaluator(config)

In [ ]:
# Demo: Evaluate a single response
sample_response = 'The speed of light in a vacuum is approximately 299,792,458 meters per second.'
sample_gt = 'The speed of light in a vacuum is approximately 299,792,458 meters per second.'

demo_result = evaluator.evaluate_response(sample_response, sample_gt)
print('Demo Evaluation Result:')
print(f"  Keyword Similarity: {demo_result['keyword_similarity']}")
print(f"  Semantic Similarity: {demo_result['semantic_similarity']}")
print(f"  Entity F1: {demo_result['entity_comparison']['entity_f1']}")
print(f"  Overall NLP Score: {demo_result['overall_nlp_score']}")

In [ ]:
# Run batch evaluation on all responses
evaluations = evaluator.batch_evaluate(responses, gt_data, queries)
print(f'\nEvaluated {len(evaluations)} queries')

In [ ]:
# Save evaluations
os.makedirs('../data/results', exist_ok=True)
with open('../data/results/nlp_evaluations.json', 'w') as f:
    json.dump(evaluations, f, indent=2, default=str)
print('Evaluations saved')

## 3. Compute Scores

In [ ]:
scorer = Scorer(config)
scores_df = scorer.score_all(evaluations, queries, responses)

print(f'\nScores DataFrame shape: {scores_df.shape}')
scores_df.head(10)

In [ ]:
# Summary statistics
summary = scorer.compute_summary_statistics(scores_df)
print('\nSummary Statistics by Chatbot:')
summary

In [ ]:
# Domain breakdown
domain_breakdown = scorer.compute_domain_breakdown(scores_df)
print('\nDomain Breakdown:')
domain_breakdown